# Point Cloud to CAD-sequence

In this notebook the complete interactive pipeline for encoding point clouds into a latent space, from which DeepCAD decodes a CAD-sequence.

In [1]:
import os
import sys
import shutil
import glob
import json
import argparse
import importlib

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn.functional as F

from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import read_step_file, write_step_file

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import vec2CADsolid, create_CAD
from models.DeepCAD.utils.file_utils import ensure_dir

### Variables

Store the models in ```experiments```, a results directory will be created for each respective model.

In [9]:
model_name = "best"

### Constants

In [10]:
model_path = os.path.join("experiments", model_name) + ".pth"
results_dir = os.path.join("experiments", model_name + "_results")
if not os.path.exists(results_dir):
    os.mkdir(results_dir)
h5_file = os.path.join(results_dir, "data.h5")
cfg = ConfigAE('test', model_path="../data/latent")
latent_dim = 256

### PointNet++

In [11]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

def load_pointnet():
    sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
    model = importlib.import_module('pointnet2_cls_ssg')
    classifier = model.get_model(latent_dim, normal_channel=False)
    criterion = model.get_loss_mse()
    classifier.apply(inplace_relu) 
    
    saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = saved_model['model_state_dict']
    if 'module.' in next(iter(state_dict)):
        monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}#
    classifier.eval()
    classifier.load_state_dict(state_dict)
    print(f"Loading PointNet++ from {os.path.abspath(model_path)}")
    return classifier

### DeepCAD

In [12]:
def load_deepcad(cfg):
    tr_agent = TrainerAE(cfg)
    tr_agent.load_ckpt(cfg.ckpt)
    tr_agent.net.eval()
    return tr_agent

### Load data

In [13]:
def get_data(indices, dataset):
    pc_list = []
    lat_rep_list = []  
    pc_paths = []
    cad_seq_list = []
    pc_dir = os.path.join(results_dir, "infered_point_clouds")
    if os.path.exists(pc_dir):
        shutil.rmtree(pc_dir)
    os.mkdir(pc_dir)
    
    for i in indices:
        pc, lat_rep, cad_seq = dataset[i]
        pc_path = os.path.abspath(dataset.get_pc_path(i))
        pc_path_destination = os.path.join(pc_dir, os.path.basename(pc_path))
        shutil.copy2(pc_path, pc_path_destination)
        pc_paths.append(pc_path_destination)
        pc_list.append(pc)
        lat_rep_list.append(lat_rep)
        cad_seq_list.append(cad_seq)

    with h5py.File(h5_file, 'a') as hf:
        dt = h5py.special_dtype(vlen=str)
        path_dataset = hf.create_dataset("pc_paths", shape=(len(pc_paths),), dtype=dt)
        path_dataset[:] = pc_paths
        
    pc_batch = torch.stack(pc_list, dim=0)
    lat_rep_batch = torch.stack(lat_rep_list, dim=0)
    cad_seq_batch = torch.stack(cad_seq_list, dim=0)
    
    return pc_batch, lat_rep_batch, cad_seq_batch

### Inference

In [14]:
def infer_pointnet(indices, dataset, model):
    with h5py.File(h5_file, 'w') as hf:
        z_pred = hf.create_dataset('z_pred', 
                                   shape=(len(indices), latent_dim), 
                                   dtype=np.float32)
        z_target = hf.create_dataset('z_target',
                                     shape=(len(indices), latent_dim),
                                     dtype = np.float32)
        seq_target = hf.create_dataset('seq_target', 
                                       shape=(len(indices), cfg.max_total_len, cfg.n_args + 1), 
                                       dtype=np.int64)
        
        pc, lat_rep, cad_seq = get_data(indices, dataset)
        z_target[:] = lat_rep
        seq_target[:] = cad_seq

        criterion_loader = importlib.import_module('pointnet2_cls_ssg')
        criterion = criterion_loader.get_loss_mse()
        
        with torch.no_grad():
            pc = pc.transpose(2, 1)
            pred, _ = model(pc)
            z_pred[:] = pred.detach()
            loss = criterion(pred, lat_rep)
            print(f"Avg. MSE-Loss: {loss.detach().item():.8e}")
            return pred, cad_seq

In [35]:
def infer_deepcad(pred, cad_seq, tr_agent):
    with h5py.File(h5_file, 'a') as hf:
        seq_pred = hf.create_dataset('seq_pred', 
                                     shape=(pred.shape[0], cfg.max_total_len, cfg.n_args + 1), 
                                     dtype=np.int64)
        cmd_logits = hf.create_dataset('cmd_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_commands), 
                                       dtype=np.float32)
        args_logits = hf.create_dataset('args_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_args, cfg.args_dim + 1), 
                                       dtype=np.float32)
        with torch.no_grad():
            pred = pred.unsqueeze(dim = 1)
            output = tr_agent.decode(pred)

            output["tgt_commands"] = cad_seq[:, :, 0] 
            output["tgt_args"] = cad_seq[:, :, 1:]
            loss_dict = tr_agent.loss_func(output)
            
            batch_out_vec = tr_agent.logits2vec(output)
            
            cmd_logits[:] = output['command_logits']
            args_logits[:] = output['args_logits']
            print(batch_out_vec.shape)
            seq_pred[:] = batch_out_vec
            
            print(f"Avg. Command-Loss: {loss_dict['loss_cmd'].detach().cpu().item():.8e}")
            print(f"Avg. Argument-Loss: {loss_dict['loss_args'].detach().cpu().item():.8e}")

### Utils

In [16]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=-1, keepdims=True)

In [17]:
def cross_entropy(logits, target):
    logits = torch.from_numpy(logits).unsqueeze(0)
    target = torch.tensor([target]).long()
    return F.cross_entropy(logits, target)

# Visualization

In [18]:
def show_results(idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][idx].decode("utf-8")
        args_logits = hf['args_logits'][idx]
        cmd_logits = hf['cmd_logits'][idx]
        seq_pred = hf['seq_pred'][idx]
        seq_target = hf['seq_target'][idx]
        z_pred = hf['z_pred'][idx]
        z_target = hf['z_target'][idx]

    ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
    LINE_IDX = ALL_COMMANDS.index('Line')
    ARC_IDX = ALL_COMMANDS.index('Arc')
    CIRCLE_IDX = ALL_COMMANDS.index('Circle')
    EOS_IDX = ALL_COMMANDS.index('EOS')
    SOL_IDX = ALL_COMMANDS.index('SOL')
    EXT_IDX = ALL_COMMANDS.index('Ext')

    print(f"Point Cloud path: {pc_path}")
   # print(args_logits.shape, cmd_logits.shape, seq_pred.shape, seq_target.shape, z_pred.shape, z_target.shape)
    for idx, command in enumerate(ALL_COMMANDS):
        print(f"{idx} -> {command}")

    target_commands = []
    predicted_commands = []
    pred_commands_prob = []
    cmd_loss = []
    cmd_loss_torch = []
    target_commands_prob = []
    
    all_pred_commands = list(seq_pred[:, 0])
    seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3
    cmd_logits_softmax = softmax(cmd_logits[:seq_length, :])
    
    for i in range(seq_length):
        predicted_commands.append(int(all_pred_commands[i]))
        target_commands.append(int(seq_target[i, 0]))
        pred_commands_prob.append(round(float(cmd_logits_softmax[i, predicted_commands[i]]) * 100, 5))
        target_commands_prob
        cmd_loss.append(cross_entropy(cmd_logits[i,:], target_commands[i]).item())
        target_commands_prob.append(round(float(cmd_logits_softmax[i, target_commands[i]]) * 100, 5))
    
    df = pd.DataFrame(list(zip(target_commands, predicted_commands, target_commands_prob, pred_commands_prob, cmd_loss)),
                      columns=['trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    print(f"Sum CADLoss for {seq_length} commands:  {sum(cmd_loss):.8e}")
    print(f"Mean CADLoss for {seq_length} commands: {np.mean(cmd_loss):.8e}")
    return df

### Export to step data

In [81]:
def export2step():
    form = "h5"
    filter = True
    output_dir = os.path.join(results_dir, "step_files")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.mkdir(output_dir)
    h5_path = os.path.join(results_dir, "data.h5")

    with h5py.File(h5_path, 'r') as fp:
        out_vec = fp['seq_pred'][:].astype(np.float64)
        names = fp['pc_paths'][:]
        
        for i, seq in enumerate(out_vec):
            pc_path = names[i].decode('utf-8')
            out_shape = vec2CADsolid(seq)
    
            if filter:
                analyzer = BRepCheck_Analyzer(out_shape)
                if not analyzer.IsValid():
                    print(f"CAD-sequence of {os.path.basename(pc_path)} is invalid.")
                    continue
    
            pc_name = os.path.splitext(os.path.basename(pc_path))[0]
            save_path = os.path.join(output_dir, pc_name + ".step")
            write_step_file(out_shape, save_path)


## Start

In [89]:
pointnet_plusplus = load_pointnet()
deepcad = load_deepcad(cfg)

Loading PointNet++ from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/notebooks/experiments/best.pth
Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


In [90]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
print(f"Dataset contains {len(dataset)} samples.")

Dataset contains 8038 samples.


In [101]:
indices = [0,1234]

In [102]:
pred, trgt_cad_seq = infer_pointnet(indices, dataset, pointnet_plusplus)

Avg. MSE-Loss: 3.20507549e-02


In [103]:
infer_deepcad(pred, trgt_cad_seq, deepcad)

(2, 60, 17)
Avg. Command-Loss: 8.76535580e-07
Avg. Argument-Loss: 2.87934327e+00


In [105]:
show_results(1)

Point Cloud path: experiments/best_results/infered_point_clouds/00607195.ply
0 -> Line
1 -> Arc
2 -> Circle
3 -> EOS
4 -> SOL
5 -> Ext
Sum CADLoss for 8 commands:  6.67569793e-06
Mean CADLoss for 8 commands: 8.34462242e-07


,trgt,pred,prob_trgt,prob_pred,loss
0,4,4,99.99933,99.99933,0.000007
1,2,2,100.00000,100.00000,0.000000
2,4,4,100.00000,100.00000,0.000000
3,2,2,100.00000,100.00000,0.000000
4,5,5,100.00000,100.00000,0.000000
5,3,3,100.00000,100.00000,0.000000
6,3,3,100.00000,100.00000,0.000000
7,3,3,100.00000,100.00000,0.000000


In [106]:
export2step()


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : experiments/best_results/step_files/00250456.step(596 ents)  Write  Done

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : experiments/best_results/step_files/00607195.step(232 ents)  Write  Done


### Gedanken

- ich sollte hier ein Ordner haben in den ich das zu benutzende model setze DONE
- dort werden auch die ergebnisse/visualisierungen abgespeichert DONE
- Infer PC -> Copy all pc's into pc folder in results folder for fast access in CC, overwrite that folder in every inference DONE
- create h5py dataset and save the pred z, target z, pred CAD seq and target CAD seq DONE
- modularize even more DONE
- visulization of CAD commands DONE
- maybe function that shows the specific inference of one point cloud? DONE
- add experiments folder to gitignore DONE



Was noch wichtig/Meeting morgen:
- infer whole train, val, test set with best model CAD Loss
- test best pointnet model? DONE
- export2step DONE
- args loss visualization
- automatic input to Cloudcompare?
- Check if CC works with pc and how to import mesh
- USE BEST MODEL FROM LAST RUN!

Meeting: 
- had to finish applications
- first thing I did was refactor training
    - Automatic resume of training if cluster fails
    - Parallelization (30mins/epoch -> 12 mins/epoch)
- implemented test script
- Implemented cosine annealing learning rate -> show new training with better convergence!
- Worked on CAD Loss understanding
- Implemented testing pipeline to make sure the data is alligned
- Finished pc2cad pipeline with thorough understanding of loss

HiWi:
- created SAiL poster
- documented literature research
- made Blensor work

Next:

- train DeepCAD ourselves?
- use blensor to create new data?
- 
